# Data-Driven Bayesian Empirical Fragility Modelling

**Author(s):** Hossein Ebrahimian, Jingren Wu, Fatemeh Jalayer

**Description:**  
This Jupyter Notebook presents the workflow for data-driven Bayesian empirical fragility modelling. It illustrates how the fragility data provided in the EPOS-ICS-C portal under the layer “Empirical Tsunami Risk Products Dataset (ETRiS v0)” can be generated.

The workflow uses observed damage and flow depth data from past tsunami events. It can also incorporate seismic data from past events. The Notebook derives empirical fragility curves and their confidence bands through Bayesian inference using MCMC simulation. It applies a generalized Binomial regression model with three possible link functions: logit, probit, and cloglog.
The approach supports hierarchical fragility modelling for a set of mutually exclusive and collectively exhaustive damage states and for different classes of buildings or infrastructure. In the current version of the Notebook, empirical fragility is computed for a single building class. However, in the upcoming version, multiple building classes will be handled simultaneously to estimate their respective empirical fragilities.

The observed data include information on tsunami effects (e.g., tsunami height, flow depth) and tsunami consequences (e.g., casualties, building damage). The resulting fragility curves express the probability of exceeding a given damage state as a function of a tsunami intensity measure (e.g., flow depth) for a specific building class or, more generally, a particular asset at risk.

Each fragility model refers to a different link function. Model 1 refers to a logistic link function of the type: (1+exp(ax+b))^-1; Model 2 refers to a probit link function of the type F(ax+b), where F is the standard Gaussian CDF. Model 3 refers to a cloglog link function of the type 1-exp(-exp(ax+b)).

**References:**  
Jalayer, F., Ebrahimian, H., Trevlopoulos, K., Bradley, B. (2023). Empirical tsunami fragility modelling for hierarchical damage levels. Natural Hazards and Earth System Sciences, 23(2), 909-931. https://nhess.copernicus.org/articles/23/909/2023/

**Citation:**  
https://doi.org/10.5281/zenodo.18551506

Note: Before running the Python code, the following libraries should be installed:

In [ ]:
# Initial Libraries

import pandas as pd
import plotly.graph_objects as ply
import numpy as np
import scipy.stats as ss
import math
import functionlib as fn


The number of Damage Levels D should be defined here. The GRM link function can be 'logit', 'probit', or 'comploglog'.

In [ ]:
# Initial Data Input

D = np.arange(0, 6)

GRM = 'logit'


Since the fragility data can be attributed to different damage scales and can have different number fo damaeg levels, we first detect the number of damage levels from the csv file. The csv file has two columns. The first column is the IM values and the second colum is the associated damage levels.

In [ ]:
# Read Input CSV file and Damage State Definition

damage_data = pd.read_csv('data/Samoan Building Class 1.csv', sep=',', header=None).values 

NDS = D[-1] - D[0]

# List of length NDS+1 
DS = [None] * (NDS + 1)

for j in range(NDS + 1):
    mask = damage_data[:, 1] == D[j]      
    DS[j] = damage_data[mask, 0]         


Starting the MCMC for Chain 1

Some Notes: The prior distribution is assumed to be uniform for all fragility model parameters. We initialize the MCMC algorithm with an arbitrary proposal distribution, chosen as a multivariate normal distribution whose mean is set equal to the Maximum Likelihood Estimate (MLE) of the hierarchical fragility model parameters. The coefficient of variation (COV) for all uncertain parameters is initially set to 0.30 (sometimes a smaller value around 0.10 is suggetsed in case of low acceptance rates). This value is user‑adjustable and can be modified based on the desired level of dispersion in the proposal distribution. Chain 1 will provide a minimum of 1000 samples ("num_samples", user-adjustable), and the algorithem starts with 2000 iterations (maxIteration), and repeats until "num_samples" samples are generated. The results are stored in "seeds_1". Please wait until the execution of each chain be ended.

In [ ]:
# MCMC Chain 1

nchain = 1

DATA = [DS, GRM]
        
cov_THETA = 0.30 
THETA_MLE = fn.glm_hierarchical(DS,NDS,GRM)
THETA_MLE = THETA_MLE.squeeze()
sigma_THETA = np.abs(THETA_MLE) * cov_THETA

THETA = THETA_MLE;

num_samples = 1000
maxIteration = 2000

dummy = np.empty((THETA.shape[0], 0))  

while dummy.shape[1]<num_samples:
    state = np.zeros((len(THETA), maxIteration))
    print(f"-------- Chain {nchain}, sample size {dummy.shape[1]}")
    for iter in range(maxIteration):
        THETA = fn.postMCMC(THETA, THETA_MLE, sigma_THETA, fn.likelihoodFunction_DS, DATA)        
        state[:, iter] = THETA

    state_unique, ind = np.unique(state.T, axis=0, return_index=True)
    state = state[:, np.sort(ind)]
    dummy = np.hstack([dummy, state])
    dummy_unique, ind = np.unique(dummy.T, axis=0, return_index=True)
    dummy = dummy[:, np.sort(ind)]

print(f"-------- Chain {nchain}, final sample size {dummy.shape[1]}")
seeds_1 = dummy

Starting the MCMC for Chain 2 to 4

Some Notes: We employ an adaptive Gaussian random‑walk proposal to construct an efficient proposal distribution. The adaptation uses the samples (“seeds”) generated in the previous segment of the chain. Instead of relying on a fixed covariance structure, the algorithm learns an empirical covariance matrix from the chain history and applies an optimal scaling coefficient suitable for high‑dimensional target distributions. A small diagonal perturbation is added, when necessary, to ensure that the covariance matrix remains positive definite. The adaptation aims for an acceptance rate of approximately 0.234, which is optimal for high‑dimensional random‑walk Metropolis algorithms.
For monitoring and diagnostic purposes, the execution of the sampler is separated into Chain 2 through Chain 4. The number of generated samples, denoted as num_samples, may vary between chains depending on user specifications or convergence needs. Please wait until the execution of each chain be ended.

In [ ]:
# MCMC Chain 2

nchain = 2

THETA = seeds_1[:, -1]

seeds = seeds_1;

num_samples = 1000
maxIteration = 2000

dummy = np.empty((THETA.shape[0], 0))  

while dummy.shape[1]<num_samples:
    state = np.zeros((len(THETA), maxIteration))
    print(f"-------- Chain {nchain}, sample size {dummy.shape[1]}")
    for iter in range(maxIteration):
        THETA = fn.postMCMCupdated(THETA, seeds, fn.likelihoodFunction_DS, DATA)        
        state[:, iter] = THETA

    state_unique, ind = np.unique(state.T, axis=0, return_index=True)
    state = state[:, np.sort(ind)]
    dummy = np.hstack([dummy, state])
    dummy_unique, ind = np.unique(dummy.T, axis=0, return_index=True)
    dummy = dummy[:, np.sort(ind)]

print(f"-------- Chain {nchain}, final sample size {dummy.shape[1]}")
seeds_2 = dummy

In [ ]:
# MCMC Chain 3

nchain = 3

THETA = seeds_2[:, -1]

seeds = seeds_2;

num_samples = 1000
maxIteration = 2000

dummy = np.empty((THETA.shape[0], 0))  

while dummy.shape[1]<num_samples:
    state = np.zeros((len(THETA), maxIteration))
    print(f"-------- Chain {nchain}, sample size {dummy.shape[1]}")
    for iter in range(maxIteration):
        THETA = fn.postMCMCupdated(THETA, seeds, fn.likelihoodFunction_DS, DATA)        
        state[:, iter] = THETA

    state_unique, ind = np.unique(state.T, axis=0, return_index=True)
    state = state[:, np.sort(ind)]
    dummy = np.hstack([dummy, state])
    dummy_unique, ind = np.unique(dummy.T, axis=0, return_index=True)
    dummy = dummy[:, np.sort(ind)]

print(f"-------- Chain {nchain}, final sample size {dummy.shape[1]}")
seeds_3 = dummy

In [ ]:
# MCMC Chain 4

nchain = 4

THETA = seeds_3[:, -1]

seeds = seeds_3;

num_samples = 1500
maxIteration = 2000

dummy = np.empty((THETA.shape[0], 0))  

while dummy.shape[1]<num_samples:
    state = np.zeros((len(THETA), maxIteration))
    print(f"-------- Chain {nchain}, sample size {dummy.shape[1]}")
    for iter in range(maxIteration):
        THETA = fn.postMCMCupdated(THETA, seeds, fn.likelihoodFunction_DS, DATA)        
        state[:, iter] = THETA

    state_unique, ind = np.unique(state.T, axis=0, return_index=True)
    state = state[:, np.sort(ind)]
    dummy = np.hstack([dummy, state])
    dummy_unique, ind = np.unique(dummy.T, axis=0, return_index=True)
    dummy = dummy[:, np.sort(ind)]

print(f"-------- Chain {nchain}, final sample size {dummy.shape[1]}")
seeds_4 = dummy

In this stage, the Robust Fragility function and its associated confidence band are computed using the samples obtained from the final chain of the MCMC procedure.

Notes:
(1) The program directly estimates the confidence interval using ±1 standard deviation. However, the user may modify this interval by adjusting the last two lines of this section.
(2) This part of the notebook can also be used to visualize the fragility functions obtained from earlier chains. To do so, simply modify the first line of this section and assign "samples" to any of seeds_1, seeds_2, seeds_3, or seeds_4.

In [ ]:
# Calculate Fragility

#-------- Initial Assignments
samples = seeds_4

dIM = 0.01
IM_max = 5.0
IM = np.concatenate(([1e-6], np.arange(dIM, IM_max + dIM, dIM)))
tol = -1e-10

sample_fragility = []
sample_PDS_IM = []
rejected_samples = []

dstep = 5
count = 0

#-------- Calculate NDS fragilities for each sample
for i in range(samples.shape[1]):
    theta_i = samples[:, i]

    dummy_fragility, dummy_PDS = fn.calculate_fragility(IM, theta_i, NDS, GRM)

    # ---- Rejection conditions ----
    subset = dummy_fragility[0:len(IM):dstep, :]
    decreasing = np.any(np.diff(subset, axis=0) < tol)

    too_large_start = np.any(dummy_fragility[0, :] > 0.01)

    if decreasing or too_large_start:
        rejected_samples.append(i)
    else:
        count += 1
        sample_fragility.append(dummy_fragility)

        sample_PDS_IM.append(np.column_stack(dummy_PDS))

# Convert rejected list to numpy array
rejected_samples = np.array(rejected_samples, dtype=int)

# Remove rejected columns from sample_theta
sample_theta = np.delete(samples, rejected_samples, axis=1)

#-------- Calculate the Robust Fragility and its standard deviation
Rfragility = np.zeros((len(IM), NDS))
sfragility = np.zeros((len(IM), NDS))

sample_fragility_DSj = [None] * NDS  # list instead of cell array

for j in range(NDS):
    # Collect fragility curves for damage state j across all samples
    mat = np.zeros((len(IM), sample_theta.shape[1]))

    for i in range(sample_theta.shape[1]):
        mat[:, i] = sample_fragility[i][:, j]

    sample_fragility_DSj[j] = mat

    Rfragility[:, j], sfragility[:, j] = fn.reliability(mat)

#-------- Assign the Robust Fragility

RF = Rfragility
RF_plus1sigma = Rfragility + sfragility
RF_minus1sigma = Rfragility - sfragility

Plot the fragility curves and their confidence interval for all damage levels: 
Plot flow depth versus mean, mean minus 1 sigma, mean plus 1 sigma fragility values for all damage levels detected. This step can be repeated for as as many damage levels for which the fragility information is available.

In [ ]:
# Plot the fragility curve for all damage levels

#-------- Define the color
myColor= []                        
myColor.append("rgb(69,139,116)")  # color dark green 
myColor.append("rgb(124, 252, 0)") # color lawngreen
myColor.append("rgb(255,215,0)")   # color yellow
myColor.append("rgb(204,0,204)")   # color magenta
myColor.append("rgb(255,48,48)")   # color firebrick

#-------- Define Linetype
myDash = []
myDash.append('dot')
myDash.append('dash')
myDash.append('solid')
myDash.append('dashdot')
myDash.append('solid')

mywidth = [3.0,3.5,4.0,4.5,5.0,5.5]

damageLevel = D[1:]
intensityType = 'Flow depth [m]'

fig = ply.Figure()
i=0
while i<NDS: 
    
    i +=1

    fig.add_scatter(x=IM,y=RF_minus1sigma[:,i-1],mode='lines',line=dict(color=myColor[i-1],width=0.5),opacity=0.5,showlegend=False)
    fig.add_scatter(x=IM,y=RF_plus1sigma[:,i-1],mode='lines',line=dict(color=myColor[i-1],width=0.5),opacity=0.5,fill='tonexty',name=('±1\u03C3 confidence interval D'+str(damageLevel[i-1])))

    fig.add_scatter(x=IM,y=RF[:,i-1],mode='lines',line=dict(color=myColor[i-1],width=mywidth[i-1],dash=myDash[i-1]),name=('Mean fragility D'+str(damageLevel[i-1])))
    
    fig.update_xaxes(showline=True,linecolor='black',title=intensityType, tickmode = 'linear',tick0 = 0.0,dtick = 1,ticklen=3, tickcolor='black',ticks="inside",showgrid=True)
    fig.update_yaxes(showline=True,linecolor='black',title='Probability of exceeding damage level', tickmode = 'linear',tick0 = 0.0,dtick = 0.1,ticklen=3, tickcolor='black',ticks="inside",showgrid=True)
    
    #bgColor = 'lightgray'
    bgColor = 'rgb(189,189,189)'
    
    fig.update_layout(width=1000,height=800,plot_bgcolor=bgColor,legend=dict(bordercolor='black',borderwidth=0.5,x=1,y=0.05,xanchor='right',yanchor='bottom'),font=dict(size=20))

    fig.update_xaxes(rangemode="tozero")
    fig.update_xaxes(range=[0,5])
    fig.update_yaxes(range=[0,1.0])

    fig.update_xaxes(showspikes=True)
    fig.update_yaxes(showspikes=True)

fig.show()

Estimate the fragility parameters for all damage levels:
The median and the logarithmic standard deviation of the fragility curve for all damage levels are estimated. The median shows the flow depth with 50% probability of being exceeded. The logarithmic standard deviation is equal to half the ratio between the 84th and 16th percentile intensity measures and is a measure of the spread of the fragility curve.

The epistemic uncertainty: In this section, the epistemic uncertainty paramater can be estimated for the fragility curves corresponding to all damage level. This information can be interpreted as the uncertainty in the fragility median expressed as logarithmic standard deviation. This is roughly equal to half of (natural log of) the ratio of the median of the plus-one-sigma fragility curve to the median of the minus-one-sigma fragility curve.

In [ ]:
# Estimating the parameters of the fragility curve

# Precompute probability levels
p50 = 0.5
p16 = ss.norm.cdf(-1.0)
p84 = ss.norm.cdf(1.0)

# Helper to interpolate each column of a matrix
def interp_columns(y_target, x_vals, y_matrix, left=np.nan, right=np.nan):
    return np.array([
        np.interp(y_target, y_matrix[:, j], x_vals, left=left, right=right)
        for j in range(y_matrix.shape[1])
    ])

# --- Median and percentiles of central fragility ---
etaIMc = interp_columns(p50, IM, RF)
IMc16  = interp_columns(p16, IM, RF, left=np.nan, right=np.nan)
IMc84  = interp_columns(p84, IM, RF, left=np.nan, right=np.nan)

# --- Percentiles from ±1σ curves ---
IM16 = interp_columns(p50, IM, RF_plus1sigma)
IM84 = interp_columns(p50, IM, RF_minus1sigma)

# --- Lognormal dispersion ---
betaIMc = 0.5 * np.log(IMc84 / IMc16)

# Handle undefined percentiles
mask16_nan = np.isnan(IMc16)
mask84_nan = np.isnan(IMc84)

if np.any(mask16_nan):
    print("16th percentile cannot be defined on some fragility curves!")
    betaIMc[mask16_nan] = np.log(IMc84[mask16_nan] / etaIMc[mask16_nan])

if np.any(mask84_nan):
    print("84th percentile cannot be defined on some fragility curves!")
    betaIMc[mask84_nan] = np.log(etaIMc[mask84_nan] / IMc16[mask84_nan])

# --- Uncertainty factor dispersion ---
betaUF = 0.5 * np.log(IM84 / IM16)

Below, we can see a print of the fragility curve median (in meters), the logarithmic standard deviation (without units) and an estimate of the epistemic uncertainty in the fragility curve (without units) for all damage levels.

In [ ]:
print('Fragility statistics for damage level = ', i)
print('median =', etaIMc, ' meters')
print('logarithmic standard deviation =', betaIMc)
print('epistemic uncertainty =', betaUF)